# 02 — Kreuzvalidierung auf externen Datensaetzen

Entspricht Kapitel 5, Abschnitte *Kreuzvalidierung* und
*Clipping-Kontrollexperiment*.

Prueft, wie gut das auf SisFall trainierte Modell auf KFall, UMAFall
und FallAllD uebertraegt — jeweils ohne Anpassung (Zero-Shot) und
nach Fine-Tuning.

> **Korrektur gegenueber frueheren Laeufen:** Die Sturz-Tasks von
> KFall sind T20–T34, nicht T22–T36. Nachgewiesen anhand der
> offiziellen Labeldateien. Fruehere KFall-Ergebnisse dieses Projekts
> beruhen auf der falschen Grenze und sind nicht mehr gueltig.

## Setup

In [4]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scipy scikit-learn tensorflow openpyxl

import os, sys, shutil, time, json
from pathlib import Path
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Gemeinsame Bibliothek holen (einzige Quelle fuer alle Loader)
# fall_lib laden: zuerst GitHub, sonst Kopie auf Drive. Schlaegt beides
# fehl, wird abgebrochen -- ohne fall_lib waeren alle Ergebnisse falsch
# (andere Fensterlaenge, anderes Heruntertasten).
import urllib.request, shutil as _sh
_LIB_URL = "https://raw.githubusercontent.com/Alyelb/ML-Modell-for-Fall-detection/main/fall_lib.py"
_LIB_DRIVE = "/content/drive/MyDrive/ML_Fall_Detection/fall_lib.py"
try:
    urllib.request.urlretrieve(_LIB_URL, "/content/fall_lib.py")
    print("[fall_lib] von GitHub geladen")
except Exception as _e:
    print(f"[fall_lib] GitHub nicht erreichbar ({_e}) -- nutze Drive-Kopie")
    _sh.copy(_LIB_DRIVE, "/content/fall_lib.py")
sys.path.insert(0, "/content")
import fall_lib as fl
import importlib; importlib.reload(fl)
# Pflichtfunktionen pruefen
for _f in ("new_run", "build_windows", "znorm", "WIN_LEN"):
    assert hasattr(fl, _f), f"fall_lib unvollstaendig ({_f} fehlt) -- Datei pruefen."
print("[fall_lib] Fensterlaenge", fl.WIN_LEN, "bei", fl.FS_TARGET, "Hz")

BASE  = "/content/drive/MyDrive/ML_Fall_Detection"
fl.RESULTS_ROOT = f"{BASE}/results"
MODEL = f"{BASE}/results/fall_model.keras"      # einheitlicher Modellpfad

print("Konfiguration:", fl.default_config())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Konfiguration: {'fs_target': 50, 'window_sec': 2.0, 'overlap': 0.5, 'window_len': 100, 'step': 50, 'kfall_fall_tasks': 'T20-T34'}


In [5]:
def hole(name, ziel):
    """Kopiert einen Datensatz von Drive auf die lokale Platte.
    Drive ist fuer tausende Kleindateien zu langsam."""
    src = f"{BASE}/{name}"
    if Path(ziel).exists():
        print(f"[ok] {name} bereits lokal"); return ziel
    if not Path(src).exists():
        print(f"[!] {src} nicht gefunden. Vorhanden:")
        for p in sorted(Path(BASE).iterdir()):
            print("   ", p.name)
        return None
    t = time.time(); shutil.copytree(src, ziel)
    print(f"[ok] {name} kopiert in {time.time()-t:.0f}s")
    return ziel

In [6]:
from tensorflow import keras
model = keras.models.load_model(MODEL, compile=False)
model.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy"])
print(f"Basismodell geladen: {model.count_params():,} Parameter")
run = fl.new_run("02_transfer")
ergebnisse = {}

Basismodell geladen: 61,601 Parameter
[Lauf] /content/drive/MyDrive/ML_Fall_Detection/results/02_transfer_2026-08-26_1036


## Transferfunktion

Einheitlicher Ablauf fuer alle Datensaetze, damit die Ergebnisse
vergleichbar bleiben.

In [7]:
def transfer(trials, name):
    X, y, g, _ = fl.build_windows(trials, mode="standard")
    if len(y) == 0 or y.sum() == 0:
        print(f"[{name}] keine Sturzfenster — uebersprungen"); return None
    X = fl.znorm(X)
    tr, te = fl.subject_split(g, y, test_size=0.25)
    print(f"\n{name}: {len(y)} Fenster, {int(y.sum())} Stuerze, "
          f"{len(np.unique(g))} Probanden")
    zs = fl.evaluate(model, X[te], y[te], "Zero-Shot")
    ft_model = fl.finetune(model, X[tr], y[tr])
    ft = fl.evaluate(ft_model, X[te], y[te], "Fine-Tuned")
    return {"zero_shot": zs, "fine_tuned": ft,
            "n_windows": int(len(y)), "n_falls": int(y.sum())}

## KFall — unterer Ruecken, 100 Hz

In [8]:
KF = hole("KFALL/sensor_data_new", "/content/KFall")
LAB = hole("KFALL/label_data_new", "/content/KFall_labels")
kf = fl.load_kfall(KF, label_dir=LAB)
ergebnisse["kfall"] = transfer(kf, "KFall")

[ok] KFALL/sensor_data_new bereits lokal
[ok] KFALL/label_data_new bereits lokal
[KFall] 5075 Trials (2346 Sturz, 2729 ADL), 32 Probanden
[KFall] Sturz-Tasks T20-T34 | 2346 Aufprallzeitpunkte aus Labeldateien

KFall: 20994 Fenster, 2346 Stuerze, 32 Probanden
  Zero-Shot                   Se=1.000  Sp=0.995  Acc=0.996  FP=   22  FN=   0
  Fine-Tuned                  Se=1.000  Sp=0.998  Acc=0.998  FP=    8  FN=   0


## UMAFall — Taille und Knoechel, 20 Hz

In [9]:
UMA = hole("UMAFall Fall Detection/UMAFall_Dataset_corrected",
            "/content/UMAFall")
for pos in ["WAIST", "ANKLE"]:
    ergebnisse[f"umafall_{pos.lower()}"] = transfer(
        fl.load_umafall(UMA, position=pos), f"UMAFall {pos}")

[ok] UMAFall Fall Detection/UMAFall_Dataset_corrected bereits lokal
[UMAFall WAIST] 617 Trials (189 Sturz, 428 ADL), 18 Probanden

UMAFall WAIST: 5764 Fenster, 189 Stuerze, 18 Probanden
  Zero-Shot                   Se=0.438  Sp=0.991  Acc=0.975  FP=   15  FN=  27
  Fine-Tuned                  Se=0.917  Sp=0.986  Acc=0.984  FP=   22  FN=   4
[UMAFall ANKLE] 643 Trials (157 Sturz, 486 ADL), 19 Probanden

UMAFall ANKLE: 6486 Fenster, 157 Stuerze, 19 Probanden
  Zero-Shot                   Se=0.194  Sp=0.972  Acc=0.958  FP=   55  FN=  29
  Fine-Tuned                  Se=0.278  Sp=0.965  Acc=0.953  FP=   68  FN=  26


## FallAllD — Taille, 238 Hz

In [10]:
FA = hole("FallAllD", "/content/FallAllD")
ergebnisse["fallalld_waist"] = transfer(
    fl.load_fallalld(FA, device=3), "FallAllD WAIST")

[ok] FallAllD bereits lokal


/content/fall_lib.py:498: UserWarning: loadtxt: input contained no data: "/content/FallAllD/S09_D3_A018_T03_A.dat"
  a = np.loadtxt(acc_path, delimiter=",", dtype=np.int16)


[FallAllD WAIST] 922 Trials (254 Sturz, 668 ADL), 8 Probanden

FallAllD WAIST: 12946 Fenster, 254 Stuerze, 8 Probanden
  Zero-Shot                   Se=0.127  Sp=0.982  Acc=0.964  FP=   55  FN=  55
  Fine-Tuned                  Se=0.698  Sp=0.963  Acc=0.958  FP=  109  FN=  19


## Kontrollexperiment: Messbereich

FallAllD verwendet einen LSM9DS1 mit ±8 g, SisFall und KFall ±16 g.
Hier wird geprueft, ob das Abschneiden hoher Spitzen den
Zero-Shot-Einbruch erklaert: Die SisFall-Testdaten werden synthetisch
begrenzt und erneut bewertet.

In [11]:
sis = fl.load_sisfall(hole("SisFall_dataset", "/content/SisFall"),
                      verbose=False)
Xs, ys, gs, _ = fl.build_windows(sis, mode="standard")
_, te_s = fl.subject_split(gs, ys, test_size=0.25)
Xs_te, ys_te = Xs[te_s], ys[te_s]

peaks = fl.acc_magnitude(Xs_te).max(axis=1)
print(f"Fenster mit Spitze > 8 g: {(peaks>8).sum()}/{len(peaks)} "
      f"({100*(peaks>8).mean():.1f}%)\n")

clip = {}
for c in [16.0, 8.0, 4.0, 2.0]:
    Xc = Xs_te.copy()
    Xc[:, :, :3] = np.clip(Xc[:, :, :3], -c, c)
    clip[f"{c:g}g"] = fl.evaluate(model, fl.znorm(Xc), ys_te,
                                  f"+-{c:g} g")
ergebnisse["clipping"] = clip

[ok] SisFall_dataset kopiert in 662s
Fenster mit Spitze > 8 g: 64/12431 (0.5%)

  +-16 g                      Se=0.987  Sp=0.998  Acc=0.998  FP=   21  FN=   4
  +-8 g                       Se=0.987  Sp=0.998  Acc=0.998  FP=   21  FN=   4
  +-4 g                       Se=0.987  Sp=0.998  Acc=0.998  FP=   19  FN=   4
  +-2 g                       Se=0.963  Sp=0.999  Acc=0.998  FP=   17  FN=  11


## Zusammenfassung

In [12]:
print(f"{'Datensatz':<22} {'Zero-Shot Se':>13} {'Fine-Tuned Se':>14}")
print("-"*52)
for k, v in ergebnisse.items():
    if isinstance(v, dict) and "zero_shot" in v:
        print(f"{k:<22} {v['zero_shot']['Se']:>13.3f} "
              f"{v['fine_tuned']['Se']:>14.3f}")

fl.save_results(run, ergebnisse, config=fl.default_config(),
    notes="KFall-Sturzgrenze korrigiert auf T20-T34 (Labeldateien).")

Datensatz               Zero-Shot Se  Fine-Tuned Se
----------------------------------------------------
kfall                          1.000          1.000
umafall_waist                  0.438          0.917
umafall_ankle                  0.194          0.278
fallalld_waist                 0.127          0.698
[gesichert] /content/drive/MyDrive/ML_Fall_Detection/results/02_transfer_2026-08-26_1036/results.json


PosixPath('/content/drive/MyDrive/ML_Fall_Detection/results/02_transfer_2026-08-26_1036/results.json')